In [33]:
!pip install mlflow --quiet
!pip install pyngrok --quiet
!pip install pipreqs

# Importing the necessary libraries

In [34]:

import numpy as np
import tensorflow as tf
import random as python_random


np.random.seed(42)

python_random.seed(42)


tf.random.set_seed(42)
#import numpy as np
import pandas as pd

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import optimizers, losses, activations, models
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Flatten, MaxPooling2D, Dropout,LayerNormalization, TimeDistributed, BatchNormalization, LeakyReLU, ReLU, Dropout, TimeDistributed, Input, Convolution1D, MaxPool1D, GlobalMaxPool1D
from tensorflow.keras.optimizers import Adam, Nadam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import StratifiedKFold
import time
from keras.callbacks import ModelCheckpoint, CSVLogger
from tensorflow.keras.callbacks import TensorBoard

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from tensorflow.keras import layers
from tensorflow.keras.layers import TimeDistributed, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt

import tensorflow as tf
import os
from tensorboard.plugins.hparams import api as hp

import mlflow
import mlflow.keras
import mlflow.pyfunc
from mlflow.pyfunc import PythonModel
from mlflow.utils.environment import _mlflow_conda_env


# Setting the Experiment name for MLFlow records

In [35]:
mlflow.set_experiment('CNN_CIFAR_data')
base_path = ""

# Downloading and reading the dataset through keras built-in function

In [36]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Information regarding the number of training and testing images available

In [37]:
print("X train = ", X_train.shape)
print("y train = ", y_train.shape)
print("X test = ", X_test.shape)
print("y tesdt = ", y_test.shape)
X_train, X_test = X_train/255, X_test/255

# Function to build a model with a particular configuration(taken as input) and return the compiled model

In [38]:
def get_CNN_model(input_shape, exp):
    
    model=Sequential()
    model.add(Input(shape = input_shape, name = 'Input_layer'))

    for i in range(exp['N_CNN']):
        model.add(Conv2D(filters = exp['N_filt'][i], 
                         kernel_size=exp['kernel_size'][i], 
                         strides=exp['strides'][i], 
                         activation = exp['CNN_act'][i],
                         padding='same',
                         name = "CNN_layer_"+str(i)))
        if(exp["max_pool_yes_or_no"][i]):
            model.add(MaxPooling2D(exp['max_pool_size'][i]))
        
    
    model.add(Flatten())

    for i in range(exp['N_FC']):
        model.add(Dense(units = exp['FC_nodes'][i], 
                         activation = exp['FC_act'][i],
                        name = "Dense_layer_"+str(i)))
    
    model.add(Dense(10, activation = 'softmax', name='Output_layer'))

    model.summary()
    opt=Adam(learning_rate=1e-4)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt,
                  metrics=['acc'])
    return model

In [39]:

input_shape = X_train.shape[1:]
print(input_shape)



# Base case
## A series of experiments are conducted changing various parameters of the model, whose perfromance is well above the baseline model

In [40]:
d= {}
input_shape=(32,32,3)
d["base_line"] =  {"input_shape": input_shape,
                    "N_CNN" :1 ,  
                    "N_filt": [16], 
                    "kernel_size": [(3,3)],
                    "max_pool_yes_or_no": [0 for i in range(1)],
                    "max_pool_size": [(2,2)],
                    "strides":[(1,1)],
                    "CNN_act":['relu'],
                    
                    "N_FC": 0,
                    "FC_nodes":[],
                    "FC_act":[],
                    "epochs":300
                    }

# Experimenting with the number of Convolutional layers

In [41]:
d["exp_1"] = {"input_shape": input_shape,
            "N_CNN" :1 ,  
            "N_filt": [16], 
            "kernel_size": [(3,3)],
            
            "strides":[(1,1) for i in range(5)],
            "CNN_act":['relu'],

            "max_pool_yes_or_no": [0 for i in range(1)],
            "max_pool_size": [(2,2)],
             
            "N_FC": 1,
            "FC_nodes":[64],
            "FC_act":['relu' for i in range(1)],
            "epochs":300
            }

d["exp_2"] = {"input_shape": input_shape,
            "N_CNN" :2 ,  
            "N_filt": [16,16], 
            "kernel_size": [(3,3),(3,3)],
            
            "strides":[(1,1) for i in range(5)],
            "CNN_act":['relu'for i in range(2)],

            "max_pool_yes_or_no":[0 for i in range(5)],
            "max_pool_size": [(2,2)],
             
            "N_FC": 1,
            "FC_nodes":[64],
            "FC_act":['relu'],
            "epochs":300
            }

d["exp_3"] = {"input_shape": input_shape,
            "N_CNN" :3 ,  
            "N_filt": [16,16,32], 
            "kernel_size": [(3,3),(3,3),(3,3)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0 for i in range(5)],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 1,
            "FC_nodes":[64],
            "FC_act":['relu'],
            "epochs":300
            }

d["exp_4"] = {"input_shape": input_shape,
            "N_CNN" :4 ,  
            "N_filt": [16,16,32,64], 
            "kernel_size": [(3,3) for i in range(4)],
            
            "strides":[(1,1) for i in range(4)],
            "CNN_act":['relu' for i in range(4)],

            "max_pool_yes_or_no": [0 for i in range(4)],
            "max_pool_size": [(2,2)],
             
            "N_FC": 1,
            "FC_nodes":[64],
            "FC_act":['relu'],
            "epochs":300
            }

d["exp_5"] = {"input_shape": input_shape,
            "N_CNN" :5 ,  
            "N_filt": [16,16,32,32,64], 
            "kernel_size": [(3,3),(3,3),(3,3),(3,3),(3,3)],
            
            "strides":[(1,1) for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0 for i in range(5)],
            "max_pool_size": [(2,2)],
             
            "N_FC": 1,
            "FC_nodes":[64],
            "FC_act":['relu'],
            "epochs":300
            }



# Experimenting on number of Fully-Connected layers

In [42]:
d["exp_6"] = {"input_shape": input_shape,
            "N_CNN" :4 ,  
            "N_filt": [16,16,32,32], 
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,0,0,0,0],
            "max_pool_size": [(2,2) for i in range(3)],
             
            "N_FC": 2,
            "FC_nodes":[64,32],
            "FC_act":['relu' for i in range(2)],
            "epochs":300
            }

d["exp_7"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,16,32,32],
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,0,0,0,0],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 3,
            "FC_nodes":[64, 32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }


# Experimenting on different settings for MaxPool layer

In [43]:
d["exp_8"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,16,32,32],              
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [1,1,1,1],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }
d["exp_9"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,16,32,32],             
             "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,0,1],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }


d["exp_10"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,16,32,32],              
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,1,0],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }

# Analyzing models with different filter sizes in each CONV layer

In [44]:
d["exp_11"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,32,64,128],              
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(2,2)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,1,0],
            "max_pool_size": [(1,1) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }
d["exp_12"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,16,64,128],             
             "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,1,0],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }


d["exp_13"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [32,32,64,128],              
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,1,0],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }


d["exp_14"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,16,32,64],              
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,1,0],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }

d["exp_15"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [16,32,32,64],              
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,1,0],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC": 2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }

d["exp_16"] = {"input_shape": input_shape,
            "N_CNN" :4 ,               
            "N_filt": [32,64,128,256],              
            "kernel_size": [(3,3) for i in range(5)],
            
            "strides":[(1,1)  for i in range(5)],
            "CNN_act":['relu' for i in range(5)],

            "max_pool_yes_or_no": [0,1,1,0],
            "max_pool_size": [(2,2) for i in range(5)],
             
            "N_FC":2,
            "FC_nodes":[64,32,16],
            "FC_act":['relu' for i in range(5)],
            "epochs":300
            }

# Function to
1. call the model
2. Fit the model to training data (90% of the training set) and evaluate model using the Validation data(10% of training set)
3. evaluate the performance of the model on test set.
4. display the train_accuracy, train_loss, validation_accuracy, validation_loss

In [45]:

#tf.random.set_seed(42)
# X_train, X_val, y_train, y_val = train_test_split(X,y, 
#                                                   test_size = 0.2, random_state = 42)

#X_train, y_train = (X,y)

results = {}

def run_experiments(d):
    print(d.keys())
    legends = []
    file_name = "_"
    for key in d.keys():
        
        print("EXPERIMENT :", key)
        model = get_CNN_model(input_shape, d[key])

        es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 10)


        with mlflow.start_run(run_name = key):

        ## ...train model, log metrics/params/model...
            mlflow.tensorflow.autolog(silent=True)

            cnn_model = model.fit(x=X_train,y=y_train,
                                batch_size=64,
                                epochs=d[key]['epochs'],
                                #epochs = 2,
                                validation_split=0.1,
                                shuffle=False,
                                callbacks=[es], verbose = 2)
            #train_loss, train_acc = model.evaluate(X_test, y_test,batch_size = 64);
            #loc='center left', bbox_to_anchor=(1, 0.5)
            test_loss, test_acc = model.evaluate(X_test, y_test,batch_size = 64);
            results[key] = {'test_acc': test_acc,'test_loss':test_loss}
            
            print(results)
            legends.append(key)
            file_name = file_name + key[4:] + "_"
            plt.rcParams.update({'font.size': 5})
            plt.subplot(2,2,1)
            plt.plot(cnn_model.history['acc'], label = 'training accuracy')
            plt.title('Accuracy')
            plt.xlabel('epochs')
            plt.ylabel('accuracy')
            plt.legend(legends,loc='center right', bbox_to_anchor=(1, 0.5))
            plt.grid()

            plt.subplot(2,2,2)
            plt.plot(cnn_model.history['val_acc'], label = 'validation accuracy')
            plt.title('Validation_Accuracy')
            plt.xlabel('epochs')
            plt.ylabel('val_accuracy')
            plt.legend(legends,loc='center right', bbox_to_anchor=(1, 0.5))
            plt.grid()

            plt.subplot(2,2,3)
            plt.plot(cnn_model.history['loss'], label = 'validation accuracy')
            plt.title('Loss')
            plt.xlabel('epochs')
            plt.ylabel('loss')
            plt.legend(legends,loc='center right', bbox_to_anchor=(1, 0.5))
            plt.grid()

            plt.subplot(2,2,4)
            plt.plot(cnn_model.history['val_loss'], label = 'validation accuracy')
            plt.title('Validation_Loss')
            plt.xlabel('epochs')
            plt.ylabel('val_loss')
            plt.legend(legends,loc='center right', bbox_to_anchor=(1, 0.5))
            plt.grid()
            plt.tight_layout()
            
            plt.savefig(file_name + ".png",dpi=1200)

        ## End the run
        mlflow.end_run()

    

# Experimenet base_line and Experiments 1 to 5

In [46]:
exp_list = ["base_line"] + ["exp_"+str(i) for i in range(1,6)]

#exp_list = ["exp_"+str(i) for i in range(6,8)]

print(exp_list)
x = {}
for i in exp_list:
    x[i] = d[i]
    
print("CURRENT EXPERIMENTS: ", x.keys())

run_experiments(x)

print(results)

with open("results_8_10.txt", 'w') as f: 
    for key, value in results.items(): 
        f.write('%s:%s\n' % (key, value))

# Experiment 6 and 7


In [47]:
#exp_list = ["base_line"] + ["exp_"+str(i) for i in range(1,6)]

exp_list = ["exp_"+str(i) for i in range(6,8)]

print(exp_list)
x = {}
for i in exp_list:
    x[i] = d[i]
    
print("CURRENT EXPERIMENTS: ", x.keys())

run_experiments(x)

print(results)

with open("results_8_10.txt", 'w') as f: 
    for key, value in results.items(): 
        f.write('%s:%s\n' % (key, value))

# Experiment 8 to 10

In [48]:
#exp_list = ["base_line"] + ["exp_"+str(i) for i in range(1,6)]

exp_list = ["exp_"+str(i) for i in range(8,11)]

print(exp_list)
x = {}
for i in exp_list:
    x[i] = d[i]
    
print("CURRENT EXPERIMENTS: ", x.keys())

run_experiments(x)

print(results)

with open("results_8_10.txt", 'w') as f: 
    for key, value in results.items(): 
        f.write('%s:%s\n' % (key, value))

# Experiment 11 to 16

In [49]:
#exp_list = ["base_line"] + ["exp_"+str(i) for i in range(1,6)]

exp_list = ["exp_"+str(i) for i in range(11,17)]

print(exp_list)
x = {}
for i in exp_list:
    x[i] = d[i]
    
print("CURRENT EXPERIMENTS: ", x.keys())

run_experiments(x)

print(results)

with open("results_11_16.txt", 'w') as f: 
    for key, value in results.items(): 
        f.write('%s:%s\n' % (key, value))



In [50]:

def get_CNN_1_model():
    model=Sequential()
    model.add(Input(shape = (32,32,3), name = 'Input_layer'))

    model.add(Conv2D(32, kernel_size = (3,3), padding='same', activation = 'relu'))
    model.add(Conv2D(64, kernel_size = (3,3), padding='same', activation = 'relu'))
    model.add(MaxPooling2D(pool_size = 2))
    
    model.add(Conv2D(128, kernel_size = (3,3),padding='same', activation = 'relu'))
    model.add(Conv2D(256, kernel_size = (3,3),padding='same', activation = 'relu'))
    model.add(MaxPooling2D(pool_size = 2))
    
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(32, activation='relu'))
    #model.add(Dense(16, activation='relu'))
    model.add(Dense(10, activation = 'softmax', name='Output_layer'))

    model.summary()
    opt=Adam(learning_rate=1e-4)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt,
                  metrics=['acc'])
    return model

In [51]:
model = get_CNN_1_model()
results={}
es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 10)
# TB = TensorBoard(log_dir=log_folder,
#                          histogram_freq=1,
#                          write_graph=True,
#                          write_images=True,
#                          update_freq='epoch',
#                          profile_batch=2,
#                          embeddings_freq=1)

with mlflow.start_run():

## ...train model, log metrics/params/model...
    mlflow.tensorflow.autolog(silent=True)

    cnn_model = model.fit(x=X_train,y=y_train,
                        batch_size=64,
                        epochs=150,
                        validation_data = (X_test,y_test),
                        shuffle=False,
                        callbacks=[],
                        verbose = 2)
    #train_loss, train_acc = model.evaluate(X_test, y_test,batch_size = 64);
    test_loss, test_acc = model.evaluate(X_test, y_test,batch_size = 64);
    results = (('test_acc', test_acc),('test_loss', test_loss))

    print(results)
    
mlflow.end_run()

# Dropout regularization

In [52]:
def get_CNN_2_model():
    model=Sequential()
    model.add(Input(shape = (32,32,3), name = 'Input_layer'))

    model.add(Conv2D(32, kernel_size = (3,3), padding='same', activation = 'relu'))
    model.add(Conv2D(64, kernel_size = (3,3), padding='same', activation = 'relu'))
    model.add(MaxPooling2D(pool_size = 2))
    model.add(Dropout(0.3))
    
    model.add(Conv2D(128, kernel_size = (3,3),padding='same', activation = 'relu'))
    model.add(MaxPooling2D(pool_size = 2))
    model.add(Dropout(0.3))
    
    model.add(Conv2D(256, kernel_size = (3,3),padding='same', activation = 'relu'))
    

    
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.4))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.2))
#     model.add(Dense(16, activation='relu'))
    #model.add(Dropout(0.1))
    model.add(Dense(10, activation = 'softmax', name='Output_layer'))

    model.summary()
    opt=Adam(learning_rate=1e-4)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt,
                  metrics=['acc'])
    return model

In [53]:
model = get_CNN_2_model()
results={}
es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 10)
# TB = TensorBoard(log_dir=log_folder,
#                          histogram_freq=1,
#                          write_graph=True,
#                          write_images=True,
#                          update_freq='epoch',
#                          profile_batch=2,
#                          embeddings_freq=1)

with mlflow.start_run():

## ...train model, log metrics/params/model...
    mlflow.tensorflow.autolog(silent=True)

    cnn_model = model.fit(x=X_train,y=y_train,
                        batch_size=64,
                        epochs=150,
                        #validation_data = (X_test,y_test),
                        shuffle=False,
                        callbacks=[],
                        verbose = 2)
    #train_loss, train_acc = model.evaluate(X_test, y_test,batch_size = 64);
    test_loss, test_acc = model.evaluate(X_test, y_test,batch_size = 64);
    results = (('test_acc', test_acc),('test_loss', test_loss))

    print(results)
    
mlflow.end_run()

In [54]:
results

# Dropout regularization + Batch normalization

In [55]:
def get_CNN_3_model():
    model=Sequential()
    model.add(Input(shape = (32,32,3), name = 'Input_layer'))

    model.add(Conv2D(32, kernel_size = (3,3), padding='same',kernel_regularizer = 'l2'))
    model.add(BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    
    model.add(Conv2D(64, kernel_size = (3,3), padding='same',kernel_regularizer = 'l2'))
    model.add(BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    
    model.add(MaxPooling2D(pool_size = 2))
    model.add(Dropout(0.4))
    model.add(Conv2D(128, kernel_size = (3,3),padding='same',kernel_regularizer = 'l2'))
    model.add(BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    
    model.add(Conv2D(256, kernel_size = (3,3),padding='same',kernel_regularizer = 'l2'))
    model.add(BatchNormalization())
    model.add(tf.keras.layers.ReLU())
    

   # model.add(Dropout(0.4))
    
    model.add(Flatten())
    model.add(Dense(64,kernel_regularizer = 'l2'))
    model.add(Dropout(0.4))
    model.add(tf.keras.layers.LeakyReLU())    
    
    
    model.add(Dense(32,kernel_regularizer = 'l2'))
    model.add(Dropout(0.2))
    model.add(tf.keras.layers.LeakyReLU())
    #
    
#      model.add(Dense(16,kernel_regularizer = 'l2'))
# #     model.add(Dropout(0.1))
#      model.add(tf.keras.layers.LeakyReLU())
    #
    
    model.add(Dense(10, activation = 'softmax', name='Output_layer'))

    model.summary()
    opt=Adam(learning_rate=1e-4)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=opt,
                  metrics=['acc'])
    return model

In [ ]:
model = get_CNN_3_model()
results={}
es = EarlyStopping(monitor = 'val_loss', mode='min', verbose = 1, patience = 10)
# TB = TensorBoard(log_dir=log_folder,
#                          histogram_freq=1,
#                          write_graph=True,
#                          write_images=True,
#                          update_freq='epoch',
#                          profile_batch=2,
#                          embeddings_freq=1)

with mlflow.start_run():

## ...train model, log metrics/params/model...
    mlflow.tensorflow.autolog(silent=True)

    cnn_model = model.fit(x=X_train,y=y_train,
                        batch_size=64,
                        epochs=300,
                        #validation_data =(X_test, y_test),
                        shuffle=False,
                        callbacks=[],
                        verbose = 2)
    #train_loss, train_acc = model.evaluate(X_test, y_test,batch_size = 64);
    test_loss, test_acc = model.evaluate(X_test, y_test,batch_size = 64);
    results = (('test_acc', test_acc),('test_loss', test_loss))

    print(results)
    
mlflow.end_run()

In [ ]:
results

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
y_test[0]
y_pred_class = np.argmax(y_pred, axis = 1)

In [ ]:
wrong_classification_list = []

for i in range(y_test.shape[0]):
    if y_test[i]!=y_pred_class[i]:
        wrong_classification_list +=[i]
        


In [ ]:
confusion_matrix(y_test,y_pred_class)
label_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
TEMP = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_class),
                               display_labels=label_classes)
plt.figure(1, figsize=(100,10))
TEMP.plot()

plt.savefig('confusion_matrix.png',dpi=1200 )

In [ ]:
len(wrong_classification_list)

In [ ]:
#f, ax =plt.subplot(4,4)
# num =0 
for i in wrong_classification_list[:10]:
        plt.imshow(X_test[i])
        plt.show()
        #print(y_test[i][0])
        print("wrong_picture_number: ",i)
        print("TRUE_CLASS: ", label_classes[y_test[i][0]] )
        print("PREDICTED_CLASS:", label_classes[y_pred_class[i]] )


# for i in range(10):
#     plt.imshow(X_test[i])
#     plt.show()    
#     print("TRUE_CLASS: ", label_classes[y_test[i][0]] )
#     print("PREDICTED_CLASS:", label_classes[y_pred_class[i]] )
#     print("\n\n\n")

In [ ]:

from pyngrok import ngrok
# Terminate open tunnels if exist
ngrok.kill()

# Setting the authtoken (optional)
# Get your authtoken from https://dashboard.ngrok.com/auth
NGROK_AUTH_TOKEN = ""
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open an HTTPs tunnel on port 5000 for http://localhost:5000
ngrok_tunnel = ngrok.connect(addr="5000", proto="http", bind_tls=True)
print("MLflow Tracking UI:", ngrok_tunnel.public_url)

!mlflow ui

In [ ]:
from keras.utils.vis_utils import plot_model
model_1 = get_CNN_1_model()
model_2 = get_CNN_2_model()
model_3 = get_CNN_3_model()

plot_model(model_1, to_file='model_1_plot.png', show_shapes=True, show_layer_names=True)
plot_model(model_2, to_file='model_2_plot.png', show_shapes=True, show_layer_names=True)
plot_model(model_3, to_file='model_3_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
!zip -r ./A2_PLOTS.zip ./

<a href="./A2_PLOTS.zip"> "./outputname.tar.gz" </a>